# Full-Range Validation on Kaggle (headless GPU)

Compares **flop-only** (flop betting, then turn+river dealt as pure runout — equity
realized, no future betting) vs **full-street** (betting on all 3 streets), full ranges.

1. kaggle.com → **Create → New Notebook** → **File → Import Notebook** (upload this file).
2. **Settings → Accelerator → GPU T4**, Internet **On**.
3. **Save Version → Save & Run All (Commit)**, then close the tab. Runs headless (~4–6 h;
   under 9 h/session and your 30 h quota).
4. Return to the finished version → open the **last cell's output** → copy `summary.json`.


In [ ]:
# 1) GPU present?
!nvidia-smi -L || echo 'NO GPU — Settings -> Accelerator -> GPU'

In [ ]:
# 2) Unpack embedded code (nothing to upload)
import base64, zipfile, io, os
_ZIP_B64 = (
    "..."
)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(_ZIP_B64))) as z: z.extractall('/kaggle/working/poker')
print('unpacked ->', sorted(os.listdir('/kaggle/working/poker')))

In [ ]:
# 3) CuPy / GPU check
try:
    import cupy as cp
except Exception:
    import subprocess,sys; subprocess.run([sys.executable,'-m','pip','install','-q','cupy-cuda12x']); import cupy as cp
nm=cp.cuda.runtime.getDeviceProperties(0)['name']; nm=nm.decode() if isinstance(nm,bytes) else nm
print('cupy',cp.__version__,'| device:',nm)

In [ ]:
# 4) Smoke: one board (confirms both models + pipeline before the long run)
!cd /kaggle/working/poker && PYTHONPATH=src python -m pokertrainer.validate_flop \
    --solver gpu --dtype float64 --n 400 --iters 200 --max-boards 1 --out /kaggle/working/smoke
import json; print(json.load(open('/kaggle/working/smoke/summary.json'))['totals'])

## Full run — FULL ranges, all 12 boards, exact (float64), 400 iters (~4–6 h headless)


In [ ]:
# 5) FULL-RANGE VALIDATION
!cd /kaggle/working/poker && PYTHONPATH=src python -m pokertrainer.validate_flop \
    --solver gpu --dtype float64 --n 400 --iters 400 --max-boards 12 --out /kaggle/working/valout

In [ ]:
# 6) Print summary.json — copy this WHOLE output back
import json; print(json.dumps(json.load(open('/kaggle/working/valout/summary.json')), indent=1))

_Per-hand CSV: `/kaggle/working/valout/flop_validation.csv` (in the committed version's Output tab)._